In [ ]:
%load_ext autoreload

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"  # Limit OpenMP
os.environ["MKL_NUM_THREADS"] = "5"  # Limit MKL (Intel Math Kernel Library)
os.environ["OPENBLAS_NUM_THREADS"] = "5"  # Limit OpenBLAS
os.environ["NUMEXPR_MAX_THREADS"] = "5"  # Limit NumExpr if installed

In [ ]:
from collections import defaultdict
import itertools
from pathlib import Path
import re
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm

In [ ]:
%autoreload 2
from src.data import get_electrode_df, add_metadata_features
from src.data_cleaning import prepare_AB_results, compute_stimulus_correlation
from src.models.decoding import run_decoding_population, run_decoding_model_comparison_population

In [ ]:
sns.set(context="paper", font_scale=2)

In [ ]:
epochs_path = "outputs/epochs_preprocessed/EC248_epo.fif"

electrodes_paths = "outputs/causal4/find_speech_responsive/EC248_results.csv"

A_result_path = Path("outputs/causal4/unify_As/results.csv")

all_A_result_path = Path("outputs/causal4/find_As/EC248_results.csv")
all_A_decoders_path = Path("outputs/causal4/find_As/EC248_decoders.pt")

B_annotated_path = Path("outputs/causal4/annotated_B_results.csv")

outdir = "outputs/causal4/behavior_decoding"

A_veridical_threshold = 0.3

In [ ]:
subject = re.findall(r"(EC[\d]+)_epo", str(epochs_path))[0]

In [ ]:
electrode_df = pd.read_csv(electrodes_paths)

In [ ]:
epochs = mne.read_epochs(epochs_path, verbose=False)
assert epochs.metadata is not None
epochs.metadata = add_metadata_features(epochs.metadata)

In [ ]:
unified_A_results, B_results = prepare_AB_results(A_result_path, B_annotated_path)

In [ ]:
B_results = B_results[B_results["subject"] == subject]

In [ ]:
A_decoders = torch.load(all_A_decoders_path)

In [ ]:
A_results = pd.read_csv(all_A_result_path).query("A and subject == @subject")

In [ ]:
A_results["stimulus_correlation"], A_outcomes = compute_stimulus_correlation(
    A_results,
    {subject: A_decoders},
    {subject: epochs},
    return_outcomes=True)

In [ ]:
A_results["veridical"] = A_results["stimulus_correlation"] > A_veridical_threshold

## Decode from manual-labeled electrodes

In [ ]:
manual_elec_labelings = {
    "stack": [
        ("EC243", 102, "bm"),
        ("EC260", 219, "bm"),
        ("EC243", 103, "bm"),
        ("EC243", 103, "pb"),
        ("EC260", 222, "pb"),
        ("EC243", 105, "bm"),
        ("EC260", 220, "pb"),
        ("EC279", 6, "dn"),
        ("EC248", 365, "dn"),
        ("EC253", 196, "dn"),
        ("EC253", 2, "dn"),
        ("EC279", 167, "dn"),
        ("EC287", 59, "dn"),
        ("EC279", 4, "bm"),
        ("EC279", 4, "dn"),
        ("EC278", 27, "bm"),
        ("EC260", 206, "bm"),
        ("EC260", 206, "pb"),
        ("EC278", 90, "dn"),
    ],

    "alligator": [
        ("EC243", 102, "dn"),
        ("EC243", 102, "pb"),
        ("EC260", 204, "dn"),
        ("EC260", 91, "dn"),
        ("EC260", 93, "dn"),
        ("EC243", 197, "bm"),
        ("EC260", 109, "dn"),
        ("EC243", 103, "dn"),
        ("EC278", 121, "bm"),
        ("EC260", 92, "dn"),
        ("EC250", 216, "pb"),
        ("EC243", 213, "dn"),
        ("EC248", 364, "dn"),
        ("EC250", 207, "dn"),
        ("EC248", 253, "dn"),
        ("EC282", 97, "dn"),
        ("EC278", 27, "dn"),
        ("EC282", 115, "pb"),
        ("EC278", 90, "pb"),
        ("EC279", 76, "bm"),
    ],

    "loo": [
        ('EC260', 204, 'dn'),
        ('EC260', 204, 'pb'),
        ('EC260', 91, 'bm'),
        ('EC260', 93, 'pb'),
        ('EC243', 197, 'dn'),
        ('EC260', 219, 'bm'),
        ('EC260', 219, 'dn'),
        ('EC278', 121, 'dn'),
        ('EC243', 119, 'bm'),
        ('EC243', 119, 'dn'),
        ('EC260', 222, 'bm'),
        ('EC260', 222, 'dn'),
        ('EC243', 105, 'pb'),
        ('EC260', 92, 'bm'),
        ('EC250', 216, 'bm'),
        ('EC260', 221, 'dn'),
        ('EC260', 221, 'pb'),
        ('EC248', 381, 'dn'),
        ('EC260', 220, 'dn'),
        ('EC253', 212, 'dn'),
        ('EC250', 215, 'bm'),
        ('EC250', 215, 'dn'),
        ('EC248', 364, 'bm'),
        ('EC260', 76, 'bm'),
        ('EC260', 76, 'dn'),
        ('EC270', 122, 'dn'),
        ('EC282', 116, 'bm'),
        ('EC287', 124, 'pb'),
        ('EC253', 196, 'pb'),
        ('EC248', 253, 'dn'),
        ('EC279', 167, 'pb'),
        ('EC279', 152, 'dn'),
        ('EC279', 152, 'pb'),
        ('EC287', 5, 'pb'),
        ('EC270', 140, 'dn'),
        ('EC260', 206, 'dn'),
        ('EC278', 90, 'bm'),
        ('EC243', 228, 'dn'),
        ('EC243', 72, 'bm'),
        ('EC243', 72, 'dn'),
        ('EC279', 11, 'pb'),
        ('EC279', 76, 'dn'),
    ]
}

In [ ]:
manual_label_df = pd.concat({label: pd.DataFrame(label_df, columns=["subject", "electrode_idx", "phoneme_pair"])
           for label, label_df in manual_elec_labelings.items()}, names=["manual_label"]).droplevel(-1).reset_index()
manual_label_df = pd.merge(manual_label_df, electrode_df, on=["subject", "electrode_idx"], how="left")

In [ ]:
manual_label_df.groupby("manual_label").roi.value_counts()

## Decode from stimulus resampled

## Decode from manually labeled electrodes

In [ ]:
manual_decoding_results = {}
for (subject_i, phoneme_pair, manual_label), rows in tqdm(manual_label_df.groupby(["subject", "phoneme_pair", "manual_label"])):
    if subject_i != subject:
        continue

    elec_idxs = rows.electrode_idx
    assert elec_idxs.nunique() == len(elec_idxs), "Duplicate electrode indices in manual labeling."
    elec_idxs = elec_idxs[elec_idxs < len(epochs.info["chs"])]
    if elec_idxs.empty:
        continue
    elec_idxs = elec_idxs.tolist()

    manual_decoding_results[subject, phoneme_pair, manual_label] = run_decoding_model_comparison_population(
        epochs,
        elec_idxs,
        phoneme_pair=phoneme_pair,
        subject=subject,
        population_name=manual_label,
        stride=10,
        window_size=30,
        target="behavior_categorical",
        baseline_features=["resampled"],
        pca_num_components=0.95,
        strategy="train-test",
        groupby=["word_end"],
    )

## Decode from A-populations, splitting by veridicality

In [ ]:
A_decoding_results = {}
for (phoneme_pair, veridical), pop in tqdm(A_results.groupby(["phoneme_pair", "veridical"])):
    A_decoding_results[subject, phoneme_pair, veridical] = run_decoding_model_comparison_population(
        epochs,
        pop.electrode_idx.tolist(),
        phoneme_pair=phoneme_pair,
        subject=subject,
        population_name="veridical" if veridical else "non-veridical",
        stride=10,
        window_size=30,
        target="behavior_categorical",
        baseline_features=["resampled"],
        pca_num_components=0.95,
        strategy="train-test",
        groupby=["word_end"],
    )

## Decode from B-populations

In [ ]:
B_decoding_results = {}
for (subject, population_name, temporal_pattern, phoneme_pair), rows in tqdm(B_results.groupby(["subject", "population_name_fixed", "Temporal pattern", "phoneme_pair"])):
    elec_idxs = rows.electrode_idx
    assert elec_idxs.nunique() == len(elec_idxs)

    result_name = f"{population_name}-{temporal_pattern}"
    B_decoding_results[subject, result_name, phoneme_pair] = run_decoding_model_comparison_population(
        epochs,
        elec_idxs,
        phoneme_pair=phoneme_pair,
        subject=subject,
        population_name=result_name,
        stride=10,
        window_size=30,
        target="behavior_categorical",
        baseline_features=["resampled"],
        pca_num_components=0.95,
        strategy="train-test",
        groupby=["word_end"],
    )

## Save

In [ ]:
torch.save({"A_decoding_results": A_decoding_results,
            "B_decoding_results": B_decoding_results,
            "manual_decoding_results": manual_decoding_results,
            },
            f"{outdir}/results.pt")